In [11]:
import keras
import numpy as np
from hgq.layers import QConv2D, QDense
from hgq.config import LayerConfigScope, QuantizerConfigScope

# First, set up quantization configuration
# For weights, use SAT_SYM overflow mode
with QuantizerConfigScope(q_type='kif', place='weight', overflow_mode='SAT_SYM', round_mode='RND'):
    # For activations, use different config
    with QuantizerConfigScope(q_type='kif', place='datalane', overflow_mode='SAT', round_mode='RND'):
        with LayerConfigScope(enable_ebops=True, beta0=1e-5):
            # Create model with quantized layers
            model = keras.Sequential([
                keras.layers.Reshape((28, 28, 1)),
                keras.layers.MaxPooling2D((2, 2)),
                QConv2D(16, (3, 3), activation='relu'),
                keras.layers.MaxPooling2D((2, 2)),
                QConv2D(32, (3, 3), activation='relu'),
                keras.layers.MaxPooling2D((2, 2)),
                keras.layers.Flatten(),
                QDense(10)
            ])

# Compile model as usual
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

In [1]:
import keras
import numpy as np
from hgq.layers import QConv2D, QDense
from hgq.config import LayerConfigScope, QuantizerConfigScope

2026-04-21 14:06:10.137192: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776773170.258518    1190 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776773170.293444    1190 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-21 14:06:10.596318: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
from hgq.utils.sugar import Dataset

input_shape = (28, 28, 1)

# Load the data and split it between train and test sets
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

# Scale images to the [0, 1] range
X_train = X_train.astype("float32") / 255
X_test = X_test.astype("float32") / 255

# Make sure images have shape (28, 28, 1)
X_train = np.expand_dims(X_train, -1)
X_test = np.expand_dims(X_test, -1)
print("X_train shape:", X_train.shape)
print(X_train.shape[0], "train samples")
print(X_test.shape[0], "test samples")

N_train = int(0.9 * len(X_train))
X_train, X_val = X_train[:N_train], X_train[N_train:]
y_train, y_val = y_train[:N_train], y_train[N_train:]

# Convert class vectors to binary class matrices
y_train = keras.utils.to_categorical(y_train, 10)
y_test = keras.utils.to_categorical(y_test, 10)
y_val = keras.utils.to_categorical(y_val, 10)

dataset_train = Dataset(X_train, y_train, batch_size=5000, device='gpu:0', shuffle=True)
dataset_val = Dataset(X_val, y_val, batch_size=5000, device='gpu:0')

X_train shape: (60000, 28, 28, 1)
60000 train samples
10000 test samples


I0000 00:00:1776773176.513690    1190 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5560 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


In [13]:
from hgq.config import LayerConfigScope, QuantizerConfigScope

with QuantizerConfigScope(
    q_type='kif', 
    place='all', 
    i0=8,            # Increase integer bits (prevents clipping/wrapping)
    f0=8,            # Increase fractional bits (prevents vanishing gradients)
    overflow_mode='SAT' # Use Saturation to keep it stable
):
    with LayerConfigScope(enable_ebops=False):
        model = keras.Sequential([
            keras.layers.Input(shape=input_shape),
            QDense(64, activation='relu'),
            QDense(64, activation='relu'),
            QDense(10)
        ])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.01),
    loss=keras.losses.CategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

In [ ]:
import keras
from hgq.layers import QDense
from hgq.config import LayerConfigScope, QuantizerConfigScope

# 1. Weights: SAT_SYM is fine
with QuantizerConfigScope(q_type='kif', place='weight', overflow_mode='SAT_SYM', round_mode='RND'):
    # 2. CRITICAL: Change WRAP to SAT for activations
    with QuantizerConfigScope(q_type='kif', place='datalane', overflow_mode='SAT', round_mode='RND'):
        # 3. Disable EBOPs initially to prove it can learn
        with LayerConfigScope(enable_ebops=False): 
            model = keras.Sequential([
                keras.Input(shape=input_shape),
                keras.layers.Flatten(),
                QDense(32, activation='relu'),
                QDense(32, activation='relu'),
                QDense(10)
            ])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss=keras.losses.CategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

# Training...
# Once accuracy hits >90%, THEN set enable_ebops=True and a small beta0

In [6]:
import keras
from hgq.layers import QDense, QConv2D
from hgq.config import LayerConfigScope, QuantizerConfigScope

   # Setup quantization configuration
   # These values are the defaults, just for demonstration purposes here
with (
      # Configuration scope for setting the default quantization type and overflow mode
      # The second configuration scope overrides the first one for the 'datalane' place
      QuantizerConfigScope(place='all', default_q_type='kbi', overflow_mode='SAT_SYM'),
      # Configuration scope for enabling EBOPs and setting the beta0 value
      QuantizerConfigScope(place='datalane', default_q_type='kif', overflow_mode='SAT'),
      LayerConfigScope(enable_ebops=True, beta0=0),
   ):
      model = keras.Sequential([
         keras.Input(shape=input_shape),
         #QConv2D(32, (3, 3), activation='relu'),
         #keras.layers.MaxPooling2D((2, 2)),
         QConv2D(32, (3, 3), activation='relu'),
         keras.layers.MaxPooling2D((2, 2)),
         keras.layers.Flatten(),
         keras.layers.Dropout(0.4),
         QDense(10)
      ])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss=keras.losses.CategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

In [14]:
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ q_dense_7 (QDense)              │ (None, 28, 28, 64)     │         2,865 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ q_dense_8 (QDense)              │ (None, 28, 28, 64)     │       167,169 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ q_dense_9 (QDense)              │ (None, 28, 28, 10)     │       153,129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 323,163 (951.59 KB)

 Trainable params: 216,948 (847.45 KB)

 Non-trainable params: 106,215 (104.14 KB)

In [15]:
from hgq.utils.sugar import FreeEBOPs
ebops = FreeEBOPs()


model.fit(dataset_train, epochs=200, validation_data=dataset_val, callbacks=[ebops], verbose=2)

Epoch 1/200


ValueError: Arguments `target` and `output` must have the same rank (ndim). Received: target.shape=(None, 10), output.shape=(None, 28, 28, 10)